# 103 — Delta-ML with Chemprop Features (CPU)

**Motivation:** nb97 uses LGBM on compressed fingerprint differences to predict delta(pEC50). Chemprop's message-passing GNN may capture richer structural context for predicting how structural changes affect activity.

**Strategy:**
1. Build the same delta-pair dataset as nb97 (Tanimoto window [0.35, 0.90])
2. Extract extra-features (x_d) for each pair: [sim, anchor_pec50, physchem_diff, compressed_fp_diff (128-d)]
3. Train a small Chemprop MPNN (depth=2, d_h=128, ffn_layers=2, 20 epochs) to predict delta using QUERY SMILES + extra features from the template anchor
4. At inference: for each test/val compound, predict delta from each template, weight by sim^2, average
5. Fall back to direct LGBM when no templates found

**CPU notes:** depth=2, d_h=128, 20 epochs keeps training time ~10min on CPU.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"): sys.stdout.reconfigure(encoding="utf-8")
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
import torch
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, compute_physchem
from pxr.paths import DATA_PROCESSED, SUBMISSIONS
import chemprop
from chemprop import data as cdata, models as cmodels, nn as cnn
SEED = 42; N_FOLDS = 5
torch.manual_seed(SEED)
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)
print(f"torch {torch.__version__} | chemprop {chemprop.__version__}")
print(f"device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

torch 2.11.0+cpu | chemprop 2.2.3
device: cpu


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if label:
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R2={r2:.4f} "
              f"r={pr:.4f} rho={sp:.4f} tau={kt:.4f}")
    return m

In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
props = ["mw","logp","tpsa","hbd","hba","rotbonds","rings"]
print("Computing physchem...", flush=True)
phys_tr = tr["smiles"].map(compute_physchem).tolist()
phys_arr = np.array([[p.get(k,0) or 0 for k in props] for p in phys_tr], dtype=np.float32)
print(f"Train {len(tr):,}  Test {len(te):,}")

Computing physchem...


Train 4,139  Test 513


In [4]:
# --- Build delta-pair training dataset ---
SIM_LO = 0.35; SIM_HI = 0.90; MAX_PAIRS = 400_000
print("Computing pairwise Tanimoto (train x train)...", flush=True)
dot_tt = (fps_tr @ fps_tr.T).astype(np.float32)
rowsum = fps_tr.sum(1).astype(np.float32)
union_tt = rowsum[:,None] + rowsum[None,:] - dot_tt
tanimoto_tr = np.where(union_tt>0, dot_tt/union_tt, 0.0)
np.fill_diagonal(tanimoto_tr, 0.0)
i_idx, j_idx = np.where((tanimoto_tr >= SIM_LO) & (tanimoto_tr <= SIM_HI))
mask_upper = i_idx < j_idx
i_idx, j_idx = i_idx[mask_upper], j_idx[mask_upper]
print(f"Pairs in sim window [{SIM_LO},{SIM_HI}]: {len(i_idx):,}")
rng = np.random.default_rng(SEED)
if len(i_idx) > MAX_PAIRS:
    sel = rng.choice(len(i_idx), MAX_PAIRS, replace=False)
    i_idx, j_idx = i_idx[sel], j_idx[sel]
    print(f"Downsampled to {MAX_PAIRS:,} pairs")

Computing pairwise Tanimoto (train x train)...


Pairs in sim window [0.35,0.9]: 5,177


In [5]:
# --- Feature engineering helpers ---
def compress_fp(fp, out_dim=64):
    N, D = fp.shape; block = D // out_dim
    return fp[:, :block*out_dim].reshape(N, out_dim, block).mean(-1).astype(np.float32)

# Extra-features per pair: [sim(1), anchor_pec50(1), physchem_diff(7), compressed_fp_diff(64)] = 73-d
# These are passed as x_d (extra atom-level descriptors at graph-level) to Chemprop
def make_xd(sim_col, anchor_pec50, phys_diff, fp_anchor, fp_query):
    fp_diff_64 = compress_fp(np.abs(fp_anchor - fp_query))
    return np.hstack([sim_col, anchor_pec50[:,None], phys_diff, fp_diff_64]).astype(np.float32)

sims_ij = tanimoto_tr[i_idx, j_idx]
phys_diff_ij = phys_arr[j_idx] - phys_arr[i_idx]
y_delta_ij = (y_tr[j_idx] - y_tr[i_idx]).astype(np.float32)

# Build both directions: (i->j) and (j->i)
xd_ij = make_xd(sims_ij[:,None], y_tr[i_idx], phys_diff_ij, fps_tr[i_idx], fps_tr[j_idx])
xd_ji = make_xd(sims_ij[:,None], y_tr[j_idx], -phys_diff_ij, fps_tr[j_idx], fps_tr[i_idx])
# Query SMILES: the compound whose pEC50 we want to predict
smiles_query_all = tr["smiles"].tolist()
query_smi_ij = [smiles_query_all[j] for j in j_idx]
query_smi_ji = [smiles_query_all[i] for i in i_idx]
xd_all = np.vstack([xd_ij, xd_ji])
y_delta_all = np.concatenate([y_delta_ij, -y_delta_ij]).astype(np.float32)
query_smi_all = query_smi_ij + query_smi_ji
print(f"Delta dataset: {len(y_delta_all):,} pairs  xd_dim={xd_all.shape[1]}")
print(f"delta range [{y_delta_all.min():.2f}, {y_delta_all.max():.2f}]")

Delta dataset: 10,354 pairs  xd_dim=73
delta range [-4.68, 4.68]


In [6]:
# --- Build Chemprop dataset for delta prediction ---
XD_DIM = xd_all.shape[1]  # 73
MAX_EPOCHS = 20
BATCH_SIZE = 256

def make_delta_dataset(smiles_list, xd_array, y_array=None):
    """MoleculeDataset where each mol is the query; anchor info lives in x_d."""
    from rdkit import Chem
    dpts = []
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            mol = Chem.MolFromSmiles("C")  # fallback
        yi = np.array([float(y_array[i])]) if y_array is not None else None
        dpts.append(cdata.MoleculeDatapoint(
            mol=mol, y=yi, x_d=xd_array[i].astype(np.float32)
        ))
    return cdata.MoleculeDataset(dpts)

def make_delta_mpnn(xd_dim, depth=2, d_h=128, ffn_layers=2, dropout=0.1):
    return cmodels.MPNN(
        message_passing=cnn.BondMessagePassing(depth=depth, d_h=d_h),
        agg=cnn.MeanAggregation(),
        predictor=cnn.RegressionFFN(
            n_tasks=1,
            n_layers=ffn_layers,
            dropout=dropout,
            input_dim=d_h + xd_dim,
        ),
    )

print(f"Chemprop delta MPNN: depth=2, d_h=128, xd_dim={XD_DIM}, max_epochs={MAX_EPOCHS}")

Chemprop delta MPNN: depth=2, d_h=128, xd_dim=73, max_epochs=20


In [7]:
# --- Train global Chemprop delta model ---
print("Building global delta dataset...", flush=True)
ds_delta = make_delta_dataset(query_smi_all, xd_all, y_delta_all)
dl_delta = cdata.build_dataloader(ds_delta, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
print(f"Dataset size: {len(ds_delta):,}  batches: {len(dl_delta)}", flush=True)

delta_mpnn = make_delta_mpnn(XD_DIM)
print(f"MPNN params: {sum(p.numel() for p in delta_mpnn.parameters()):,}", flush=True)

trainer_delta = L.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="cpu",
    enable_progress_bar=True,
    enable_model_summary=False,
    logger=False,
)
trainer_delta.fit(delta_mpnn, dl_delta)
print("Global Chemprop delta model trained.", flush=True)

Building global delta dataset...


Dataset size: 10,354  batches: 41


MPNN params: 204,321


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


...

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Loading `train_dataloader` to estimate number of stepping batches.


Output()

`Trainer.fit` stopped: `max_epochs=20` reached.


Global Chemprop delta model trained.


In [8]:
# --- Chemprop delta prediction function ---
K_NEIGHBORS = 10

def chemprop_delta_predict(smiles_query, fps_query, fps_ref, y_ref, phys_query, phys_ref,
                            sim_matrix, delta_model, trainer, direct_preds,
                            sim_lo=SIM_LO, sim_hi=SIM_HI, k=K_NEIGHBORS):
    """
    For each query compound, find templates in sim window and predict delta via Chemprop.
    Falls back to direct_preds if no templates found.
    """
    from rdkit import Chem
    N = len(smiles_query)
    preds = np.full(N, np.nan)
    n_templates = np.zeros(N, dtype=int)

    # Collect all (query_idx, template_indices) pairs
    query_batch_smi = []
    query_batch_xd  = []
    query_batch_sims = []
    query_batch_anc_pec50 = []
    pair_to_query = []  # which query does each pair belong to

    for qi in range(N):
        sim_row = sim_matrix[qi]
        cand_mask = (sim_row >= sim_lo) & (sim_row <= sim_hi)
        cand_idx = np.where(cand_mask)[0]
        if len(cand_idx) == 0:
            preds[qi] = direct_preds[qi]
            continue
        top_k = np.argsort(-sim_row[cand_idx])[:k]
        cand_idx = cand_idx[top_k]
        cand_sims = sim_row[cand_idx]
        n_templates[qi] = len(cand_idx)
        for ri, (ci, cs) in enumerate(zip(cand_idx, cand_sims)):
            phys_d = phys_query[qi] - phys_ref[ci]
            fp_d = compress_fp(np.abs(fps_query[qi:qi+1] - fps_ref[ci:ci+1]))[0]
            xd_i = np.concatenate([[cs], [float(y_ref[ci])], phys_d, fp_d]).astype(np.float32)
            query_batch_smi.append(smiles_query[qi])
            query_batch_xd.append(xd_i)
            query_batch_sims.append(cs)
            query_batch_anc_pec50.append(float(y_ref[ci]))
            pair_to_query.append((qi, cs))

    if len(query_batch_smi) == 0:
        return preds, n_templates

    # Batch predict deltas
    xd_batch = np.stack(query_batch_xd)
    ds_batch = make_delta_dataset(query_batch_smi, xd_batch)
    dl_batch = cdata.build_dataloader(ds_batch, batch_size=512, shuffle=False, num_workers=0)
    raw = trainer.predict(delta_model, dl_batch)
    delta_preds = torch.cat(raw).numpy().flatten()

    # Aggregate: weighted average per query
    anc_arr = np.array(query_batch_anc_pec50)
    sims_arr = np.array(query_batch_sims)
    qi_arr = np.array([p[0] for p in pair_to_query])
    template_preds = anc_arr + delta_preds

    for qi in range(N):
        mask = qi_arr == qi
        if not mask.any():
            continue
        w = sims_arr[mask] ** 2
        preds[qi] = np.average(template_preds[mask], weights=w)

    return preds, n_templates

# Predict-only trainer (no training, just inference)
trainer_infer = L.Trainer(
    accelerator="cpu", enable_progress_bar=False, enable_model_summary=False, logger=False
)
print(f"Chemprop multi-template predict function ready (K={K_NEIGHBORS})")

GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Chemprop multi-template predict function ready (K=10)


In [9]:
# --- Scaffold 5-fold CV ---
print("\n=== Scaffold 5-fold CV ===", flush=True)
oof_cp_delta = np.full(len(y_tr), np.nan)
oof_direct   = np.full(len(y_tr), np.nan)

for fold, (tr_idx, va_idx) in enumerate(splits):
    # Direct LGBM
    m_dir = lgb.train(LGBM, lgb.Dataset(X_tr[tr_idx], label=y_tr[tr_idx]),
                      valid_sets=[lgb.Dataset(X_tr[va_idx], label=y_tr[va_idx])],
                      callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_direct[va_idx] = m_dir.predict(X_tr[va_idx])

    # Similarity matrix: val x fold-train
    fps_va = fps_tr[va_idx]; fps_ft = fps_tr[tr_idx]
    dot_vf = (fps_va @ fps_ft.T).astype(np.float32)
    rs_v = fps_va.sum(1)[:,None]; rs_f = fps_ft.sum(1)[None,:]
    sim_vf = dot_vf / np.maximum(rs_v + rs_f - dot_vf, 1e-6)

    smi_va = [tr["smiles"].iloc[i] for i in va_idx]
    phys_va = phys_arr[va_idx]; phys_ft = phys_arr[tr_idx]

    preds_cp, n_tmpl = chemprop_delta_predict(
        smi_va, fps_va, fps_ft, y_tr[tr_idx], phys_va, phys_ft,
        sim_vf, delta_mpnn, trainer_infer, oof_direct[va_idx]
    )
    oof_cp_delta[va_idx] = preds_cp

    r_dir = rae(y_tr[va_idx], oof_direct[va_idx])
    r_cp  = rae(y_tr[va_idx], oof_cp_delta[va_idx])
    print(f"  fold {fold+1}  direct={r_dir:.4f}  cp_delta={r_cp:.4f}  "
          f"avg_templates={float(n_tmpl.mean()):.1f}", flush=True)

m_dir = full_metrics(y_tr, oof_direct,   label="direct_lgbm")
m_cp  = full_metrics(y_tr, oof_cp_delta, label="chemprop_delta")


=== Scaffold 5-fold CV ===


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


  fold 1  direct=0.4982  cp_delta=0.5272  avg_templates=1.9


  fold 2  direct=0.5759  cp_delta=0.6089  avg_templates=1.8


  fold 3  direct=0.6021  cp_delta=0.6422  avg_templates=1.7


  fold 4  direct=0.5665  cp_delta=0.5976  avg_templates=1.7


  fold 5  direct=0.6033  cp_delta=0.6407  avg_templates=1.7


  [direct_lgbm] RAE=0.5643 MAE=0.5134 R2=0.5991 r=0.7740 rho=0.7268 tau=0.5345
  [chemprop_delta] RAE=0.5981 MAE=0.5442 R2=0.5794 r=0.7639 rho=0.7236 tau=0.5310


In [10]:
# --- Blend sweep ---
best_alpha, best_rae_v = 0.0, full_metrics(y_tr, oof_direct)["RAE"]
for alpha in np.arange(0.0, 1.05, 0.1):
    blended = alpha*oof_cp_delta + (1-alpha)*oof_direct
    mask = np.isfinite(blended)
    r = rae(y_tr[mask], blended[mask])
    print(f"  alpha={alpha:.1f}  RAE={r:.4f}")
    if r < best_rae_v:
        best_rae_v, best_alpha = r, alpha

oof = best_alpha*oof_cp_delta + (1-best_alpha)*oof_direct
m_blend = full_metrics(y_tr, oof, label=f"blend(a={best_alpha:.1f})")
print(f"\nBest blend alpha={best_alpha:.1f}  OOF RAE={best_rae_v:.4f}")
print("\n" + pd.DataFrame([m_dir, m_cp, m_blend],
                           index=["direct","cp_delta",f"blend_{best_alpha:.1f}"]).round(4).to_string())

  alpha=0.0  RAE=0.5643
  alpha=0.1  RAE=0.5607
  alpha=0.2  RAE=0.5587
  alpha=0.3  RAE=0.5585
  alpha=0.4  RAE=0.5602
  alpha=0.5  RAE=0.5635
  alpha=0.6  RAE=0.5682
  alpha=0.7  RAE=0.5742
  alpha=0.8  RAE=0.5812
  alpha=0.9  RAE=0.5892
  alpha=1.0  RAE=0.5981
  [blend(a=0.3)] RAE=0.5585 MAE=0.5082 R2=0.6115 r=0.7828 rho=0.7384 tau=0.5457

Best blend alpha=0.3  OOF RAE=0.5585

              RAE     MAE      R2  Pearson  Spearman  Kendall
direct     0.5643  0.5134  0.5991   0.7740    0.7268   0.5345
cp_delta   0.5981  0.5442  0.5794   0.7639    0.7236   0.5310
blend_0.3  0.5585  0.5082  0.6115   0.7828    0.7384   0.5457


In [11]:
# --- Final test predictions ---
print("\nFitting final direct LGBM on all train...", flush=True)
m_final = lgb.train(LGBM, lgb.Dataset(X_tr, label=y_tr), callbacks=[lgb.log_evaluation(-1)])
te_direct = m_final.predict(X_te)

# Test-train similarity
dot_tet = (fps_te @ fps_tr.T).astype(np.float32)
rs_te = fps_te.sum(1)[:,None]; rs_tr_v = fps_tr.sum(1)[None,:]
sim_te_tr = dot_tet / np.maximum(rs_te + rs_tr_v - dot_tet, 1e-6)

phys_te = np.array([[p.get(k,0) or 0 for k in props]
                     for p in te["smiles"].map(compute_physchem)], dtype=np.float32)
smi_te = te["smiles"].tolist()

print("Running Chemprop delta on test...", flush=True)
te_cp, n_tmpl_te = chemprop_delta_predict(
    smi_te, fps_te, fps_tr, y_tr, phys_te, phys_arr,
    sim_te_tr, delta_mpnn, trainer_infer, te_direct
)
print(f"Test: avg templates={n_tmpl_te.mean():.1f}  min={n_tmpl_te.min()}  max={n_tmpl_te.max()}")

te_preds = best_alpha*te_cp + (1-best_alpha)*te_direct
te_preds = np.clip(te_preds, y_tr.min()-0.5, y_tr.max()+0.5)

np.save(DATA_PROCESSED/"oof_delta_chemprop_cpu.npy", oof)
np.save(DATA_PROCESSED/"te_oof_delta_chemprop_cpu.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"103_delta_chemprop_cpu.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")
print(f"\n*** nb103 OOF RAE = {m_blend['RAE']:.4f} ***")


Fitting final direct LGBM on all train...


Running Chemprop delta on test...


Test: avg templates=3.1  min=0  max=10
Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\103_delta_chemprop_cpu.csv
Test: min=2.51 med=4.88 max=5.89

*** nb103 OOF RAE = 0.5585 ***
